In [1]:
!git clone https://github.com/nhatvu205/vi-multimodal-sacarsm-detection-on-social-media.git \
    /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout nvu

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1439, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 1439 (delta 222), reused 234 (delta 164), pack-reused 1095 (from 1)
Receiving objects: 100% (1439/1439), 33.03 MiB | 27.45 MiB/s, done.
Resolving deltas: 100% (806/806), done.
/kaggle/working/repo
Branch 'nvu' set up to track remote branch 'nvu' from 'origin'.
Switched to a new branch 'nvu'


In [2]:
!pip install -q -r experiment_setup/requirements.txt
!pip install -q pillow transformers torch torchvision --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.7/440.7 kB 8.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4

In [3]:
!nvidia-smi
import torch
print(torch.cuda.is_available())

Sun May 31 13:36:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q \
    "PyYAML>=6.0" \
    "Pillow>=10.0.0" \
    "scikit-learn>=1.3.0" \
    "transformers>=4.57.0" \
    "accelerate>=0.29.0" \
    "safetensors>=0.4.0" \
    "visonorm>=0.1.4" \
    timm

In [5]:
filepath = "/kaggle/working/repo/experiment_setup/src/data.py"

with open(filepath, "r") as f:
    content = f.read()

content = content.replace(
    "RAW_IMAGE_SCENARIOS = {'s1', 's2'}\nPREPROCESSED_IMAGE_SCENARIOS = {'s3', 's4'}",
    "PREPROCESSED_IMAGE_SCENARIOS = {'s1', 's2', 's3', 's4'}"
)

with open(filepath, "w") as f:
    f.write(content)

# Verify
with open(filepath, "r") as f:
    for i, line in enumerate(f, 1):
        if "SCENARIOS" in line:
            print(f"Line {i}: {line.strip()}")

In [6]:
import torch, json
from PIL import Image
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0))

# 1. Định nghĩa lại các đường dẫn gốc cho chuẩn
DATA_ROOT = Path("/kaggle/input/datasets/nhatvu205/sacasm-dataset-uit")
IMAGES_DIR = DATA_ROOT / "images"  # Thư mục ảnh thực tế bạn vừa tìm được

# Đọc file train.json
with open(DATA_ROOT / "final-data/train.json") as f:
    data = json.load(f)
print(f"Train samples: {len(data)}")
print(f"Keys: {list(data[0].keys())}")

# 2. Sửa đoạn này: Lấy tên file ảnh rồi nối với IMAGES_DIR
# Dùng data[0]['image_path'] hoặc data[0]['image'] tùy thuộc vào key chính xác trong dữ liệu của bạn
img_filename = Path(data[0]['image_path']).name  
img_path = IMAGES_DIR / img_filename

print(f"Image path: {img_path}")
print(f"Exists: {img_path.exists()}")

# Mở ảnh
if img_path.exists():
    img = Image.open(img_path).convert("RGB")
    print(f"Size: {img.size}, Mode: {img.mode}")
    print("✅ Data OK!")
else:
    print("❌ Vẫn có lỗi xảy ra, hãy kiểm tra lại key 'image_path' hoặc tên file!")

GPU: Tesla T4
Train samples: 5884
Keys: ['id', 'text', 'image_path', 'ocr_text', 'mm_label', 'text_label', 'image_label', 'source']
Image path: /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/images/post03261.jpg
Exists: True
Size: (1440, 1920), Mode: RGB
✅ Data OK!


In [7]:
filepath = "/kaggle/working/repo/experiment_setup/src/data.py"

with open(filepath, "r") as f:
    content = f.read()

# Đoạn code cũ trong file của bạn
old_code = """def resolve_image_path(raw_path: str, config: dict) -> Path:
    raw = Path(str(raw_path))
    root = repo_root(config)
    image_root = Path(config['data'].get('image_root', '.'))
    if not image_root.is_absolute():
        image_root = root / image_root

    candidate = image_root / 'images' / raw.name"""

# Đoạn code mới an toàn tuyệt đối, ép thẳng về folder images chuẩn của Kaggle
new_code = """def resolve_image_path(raw_path: str, config: dict) -> Path:
    raw = Path(str(raw_path))
    
    # Ép cố định đường dẫn chuẩn bất chấp file YAML base có bị lồng thư mục hay không
    candidate = Path('/kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/images') / raw.name"""

if old_code in content:
    content = content.replace(old_code, new_code)
    print("✅ Đã sửa và ép đường dẫn ảnh chuẩn thành công!")
else:
    # Nếu không khớp ký tự thụt lề (space), ta dùng cách thay thế linh hoạt hơn
    import re
    pattern = r"def resolve_image_path\(raw_path: str, config: dict\) -> Path:.*?candidate = image_root / 'images' / raw\.name"
    content, count = re.subn(pattern, new_code, content, flags=re.DOTNET)
    if count > 0:
        print("✅ Đã sửa thành công bằng Regex!")
    else:
        print("❌ Không tìm thấy hàm để sửa, hãy kiểm tra lại file data.py")

# Ghi lại nội dung mới vào file data.py
with open(filepath, "w") as f:
    f.write(content)

✅ Đã sửa và ép đường dẫn ảnh chuẩn thành công!


In [8]:
!python -m experiment_setup.main \
    --config experiment_setup/configs/models/vit_b32.yaml \
    --stage preprocess \
    --json_splits \
        /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/train.json \
        /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/dev.json \
        /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/test.json \
    --image_root /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/data

[run] output dir: /kaggle/working/repo/experiment_setup/runs/vit_b32_mm
[cache] preparing train split...
2026-05-31 13:36:32,220 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-05-31 13:36:32,221 - datasets - INFO - JAX version 0.7.2 available.
2026-05-31 13:36:35,439 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-05-31 13:36:35,600 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-31 13:36:35,601 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-31 13:36:35,618 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c2475886/config.json "HTTP/1.1 200 OK"
2026-05-31 13:36:35,

In [9]:
filepath = "/kaggle/working/repo/experiment_setup/src/runner.py"

with open(filepath, "r") as f:
    content = f.read()

# Đoạn code cũ gọi hàm lưu summary
old_code = """    _save_summary(run_dir, model_name, summary_rows)"""

# Đoạn code mới: duyệt qua từng hàng kết quả và loại bỏ trường confusion_matrix trước khi ghi file
new_code = """    # Vá lỗi: Loại bỏ confusion_matrix vì file CSV không hỗ trợ lưu dict/list lồng nhau
    for row in summary_rows:
        if 'confusion_matrix' in row:
            row.pop('confusion_matrix')
    _save_summary(run_dir, model_name, summary_rows)"""

if old_code in content:
    content = content.replace(old_code, new_code)
    print("✅ Đã vá lỗi ghi file CSV thành công!")
else:
    # Nếu không khớp ký tự thụt lề, ta dùng cách tìm kiếm linh hoạt hơn
    import re
    content, count = re.subn(r"(_save_summary\(run_dir,\s*model_name,\s*summary_rows\))", new_code, content)
    if count > 0:
        print("✅ Đã vá lỗi thành công bằng cấu trúc nâng cao!")
    else:
        print("❌ Không tìm thấy hàm _save_summary để sửa. Bạn hãy kiểm tra lại file runner.py")

with open(filepath, "w") as f:
    f.write(content)

✅ Đã vá lỗi ghi file CSV thành công!


In [10]:
filepath = "/kaggle/working/repo/experiment_setup/src/runner.py"

with open(filepath, "r") as f:
    content = f.read()

# Đoạn code in log cũ trong file runner.py của bạn
old_print_code = """            print(
                f"[run] {model_name} {scenario} {split} | "
                f"acc={metrics_row['accuracy']:.4f} f1w={metrics_row['f1_weighted']:.4f}"
            )"""

# Đoạn code mới lấy chính xác các key từ metrics.py: accuracy, f1_macro, f1_weighted, precision_weighted, recall_weighted, auc
new_print_code = """            acc = metrics_row.get('accuracy', 0.0)
            f1_m = metrics_row.get('f1_macro', 0.0)
            f1_w = metrics_row.get('f1_weighted', 0.0)
            prec = metrics_row.get('precision_weighted', 0.0)
            rec = metrics_row.get('recall_weighted', 0.0)
            auc = metrics_row.get('auc', None)

            auc_str = f" auc={auc:.4f}" if auc is not None else " auc=None"
            
            print(
                f"[run] {model_name} {scenario} {split} | "
                f"acc={acc:.4f} f1_m={f1_m:.4f} f1_w={f1_w:.4f} prec={prec:.4f} rec={rec:.4f}{auc_str}"
            )"""

if old_print_code in content:
    content = content.replace(old_print_code, new_print_code)
    print("✅ Đã đồng bộ và vá hàm print thành công!")
else:
    # Sử dụng Regex phòng trường hợp khoảng trắng (indentation) trong file runner.py của bạn hơi lệch
    import re
    pattern = r"print\(\s*f\"\[run\].*?f\"acc=\{metrics_row.*?\s*\)"
    content, count = re.subn(pattern, new_print_code, content, flags=re.DOTNET)
    if count > 0:
        print("✅ Đã vá hàm print thành công bằng Regex nâng cao!")
    else:
        # Nếu vẫn không được, ta chơi bài "duyệt và thay thế" theo dòng chữ đặc trưng
        if "f1_weighted" in content:
            # Cắt và thay thế thủ công đoạn print dựa vào từ khóa f1_weighted
            lines = content.split('\n')
            for idx, line in enumerate(lines):
                if "f1_weighted" in line and "print" in lines[idx-2]:
                    # Thay thế cả cụm print cũ bằng cụm mới (giữ nguyên thụt lề 12 spaces)
                    indent = "            "
                    lines[idx-2] = indent + "acc = metrics_row.get('accuracy', 0.0)"
                    lines[idx-1] = indent + "f1_m = metrics_row.get('f1_macro', 0.0)"
                    lines[idx] = indent + "f1_w = metrics_row.get('f1_weighted', 0.0)\\n" + indent + "prec = metrics_row.get('precision_weighted', 0.0)\\n" + indent + "rec = metrics_row.get('recall_weighted', 0.0)\\n" + indent + "auc = metrics_row.get('auc', None)\\n" + indent + "auc_str = f' auc={auc:.4f}' if auc is not None else ' auc=None'"
                    lines[idx+1] = indent + "print(f'[run] {model_name} {scenario} {split} | acc={acc:.4f} f1_m={f1_m:.4f} f1_w={f1_w:.4f} prec={prec:.4f} rec={rec:.4f}{auc_str}')"
                    lines[idx+2] = "" # Xoá dòng đóng ngoặc thừa
            content = '\n'.join(lines)
            print("✅ Đã dùng giải pháp ép cấu trúc dòng để sửa thành công!")

with open(filepath, "w") as f:
    f.write(content)

✅ Đã đồng bộ và vá hàm print thành công!


In [11]:
!python -m experiment_setup.main \
    --config experiment_setup/configs/models/vit_b32.yaml \
    --stage run \
    --scenario s1 \
    --eval_splits dev test \
    --json_splits \
        /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/train.json \
        /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/dev.json \
        /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/test.json \
    --image_root /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit

[run] output dir: /kaggle/working/repo/experiment_setup/runs/vit_b32_mm
[run] model=vit-b32 scenario=s1
config.json: 100%|█████████████████████████████| 502/502 [00:00<00:00, 1.95MB/s]
model.safetensors: 100%|██████████████████████| 352M/352M [00:03<00:00, 109MB/s]
Loading weights: 100%|██████████████████████████| 6/6 [00:00<00:00, 8689.86it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch32-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight 